## DDL Gold: pf.gold.dim_tag  (SCD Type 1)
## Catalogo de tags para analitica de tags (via silver.model_tag)

In [0]:
%sql

DROP TABLE IF EXISTS pf.gold.dim_tag;

CREATE TABLE IF NOT EXISTS pf.gold.dim_tag (
    tag_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'PK - Surrogate Key',
    tag STRING NOT NULL COMMENT 'BK - texto del tag',
    tipo_tag STRING COMMENT 'license | dataset | arxiv | generico',
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'UTC',
    PRIMARY KEY (tag_id),
    CONSTRAINT uniq_dim_tag UNIQUE (tag)
)
USING DELTA
TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Dimension Tag - SCD Type 1';

In [0]:
%sql

-- Carga inicial (SCD1: inserta tags nuevos, no actualiza historial de los existentes)
MERGE INTO pf.gold.dim_tag AS t
USING (
    SELECT tag,
           CASE WHEN es_license THEN 'license'
                WHEN es_dataset THEN 'dataset'
                WHEN es_arxiv   THEN 'arxiv'
                ELSE 'generico' END AS tipo_tag
    FROM pf.silver.model_tag
    -- solo traemos tags del snapshot mas reciente para no duplicar trabajo
    WHERE ingestion_date = (SELECT MAX(ingestion_date) FROM pf.silver.model_tag)
) AS s
ON t.tag = s.tag
WHEN MATCHED THEN
    UPDATE SET t.tipo_tag = s.tipo_tag
WHEN NOT MATCHED THEN
    INSERT (tag, tipo_tag, _createdAt)
    VALUES (s.tag, s.tipo_tag, CURRENT_TIMESTAMP());

In [0]:
%sql

SELECT tipo_tag, COUNT(*) AS n_tags FROM pf.gold.dim_tag GROUP BY tipo_tag ORDER BY n_tags DESC;